# MiniGPT-Stack T4 inference benchmark (Phase 5, M5) - THIN, all logic lives in bench/

Protocol: `docs/benchmark_protocol.md` (same procedure as the CPU run): fixed generation length,
identical prompts per config, warm-up, interleaved repeats, median + IQR, `cuda.synchronize`,
paired-bootstrap speedup CIs, correctness gate BEFORE timing.

- Configs: fp32 {no cache, tuple cache, static cache} and fp16 static cache; n in {1, 8, 32}; short and
  long prompts. No-cache runs only for short n=1, short n=8, long n=1 (the rest is hours of GPU).
- **int8 is skipped** on purpose: the weight-only int8 path dequantizes on every call, it was slower on
  the CPU, and it would add roughly 15 GPU-minutes. Its value (file size 0.25x, +0.04% ppl) is already
  measured.
- Skipped on purpose: the long Phase 4 tests. Only `test_model`, `test_inference`, `test_speed` run.
- Hard cap 45 minutes (per-subprocess timeouts) and a benchmark time budget that stops starting new
  repeats (>= 2 kept) so a slow GPU produces fewer repeats, flagged, instead of no result.

**Attach two datasets:** `glassbox-dpo-init` (SFT weights, byte- and SHA-256-checked) and
`tinystories-shards-8192` (validation prompts).


In [ ]:
# ---- PINNED ----
PROMPTS   = 4          # K: identical for every config
GEN       = 128        # G: fixed generation length (EOT ignored)
REPEATS   = 10         # R: interleaved repeats
WARMUP    = 3
NS        = "1,8,32"
MODELS    = "fp32,fp16"
GRID      = "reduced"  # no-cache cells: short n=1, short n=8, long n=1 only

WEIGHTS_FILE   = "sft_final_best.pt"
WEIGHTS_BYTES  = 125_893_429
WEIGHTS_SHA256 = "d043156ee2d23aed1336cb42c80efe8cd603c40421a469d4d9519d230cbc3a5c"

import time
T0 = time.time()
HARD_CAP_S = 45 * 60            # hard cap for the whole kernel
BENCH_BUDGET_S = 28 * 60        # bench.speed stops starting repeats after this
COMPLETED = False


def time_left():
    return HARD_CAP_S - (time.time() - T0)


def capped(argv):
    """subprocess.run with the remaining hard-cap budget as its timeout (TimeoutExpired ends the kernel)."""
    import subprocess
    left = time_left()
    if left < 30:
        raise RuntimeError(f"[bench] hard cap: only {left:.0f}s left; not starting {argv[:4]}")
    return subprocess.run(argv, timeout=left)


## GPU check

In [ ]:
import torch
print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("torch", torch.__version__, "| cuda build", torch.version.cuda, "| cuda avail", torch.cuda.is_available())
assert torch.cuda.is_available(), "enable GPU (T4) in notebook settings"


## Clone

In [ ]:
import os
REPO = "https://github.com/Priyanshaur/Glassbox.git"
ROOT = "/kaggle/working/Glassbox"
if not os.path.exists(ROOT):
    !git clone $REPO $ROOT
else:
    !git -C $ROOT pull --ff-only
%cd $ROOT
!git rev-parse --short HEAD

## Install

In [ ]:
# Kaggle already ships numpy, pyyaml, tqdm, pytest, regex and datasets.
!pip install -q regex pyyaml pytest tqdm datasets


## Inputs - SFT weights, byte-exact and SHA-256 gate

In [ ]:
import glob, hashlib, os, torch

hits = glob.glob("/kaggle/input/**/" + WEIGHTS_FILE, recursive=True)
assert hits, "attach the private dataset glassbox-dpo-init (contains sft_final_best.pt)"
W = hits[0]
got = os.path.getsize(W)
print(W, "| bytes", got, "(expected", WEIGHTS_BYTES, ")")
assert got == WEIGHTS_BYTES, "weights size mismatch (truncated upload?)"
h = hashlib.sha256()
with open(W, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
print("[bench] sha256 expected ", WEIGHTS_SHA256)
print("[bench] sha256 in-kernel", h.hexdigest(), "->", "MATCH" if h.hexdigest() == WEIGHTS_SHA256 else "MISMATCH")
assert h.hexdigest() == WEIGHTS_SHA256, "SHA-256 mismatch; stopping before anything runs"

from common.config import load_config
from model.gpt import GPT, count_params
cfg = load_config("configs/base_30m.yaml")
_m = GPT(cfg["model"])
_m.load_state_dict(torch.load(W, map_location="cpu", weights_only=True))
print("weights load into the configured arch | params:", count_params(_m))
del _m
# the real-checkpoint tests (skipped when the file is absent) look for it here
os.makedirs("checkpoints/glassbox-sft-final", exist_ok=True)
link = "checkpoints/glassbox-sft-final/sft_final_best.pt"
if os.path.lexists(link):
    os.remove(link)
os.symlink(W, link)


## Shards (validation prompts)

In [ ]:
import glob, json, os
hits = glob.glob("/kaggle/input/**/meta.json", recursive=True)
assert hits, "attach the shards dataset first"
print("meta.json candidates:", hits)
shard_dir = os.path.dirname(hits[0])
print("--- meta.json head ---")
print("\n".join(open(os.path.join(shard_dir, "meta.json")).read().splitlines()[:15]))
os.makedirs("data", exist_ok=True)
# The cloned repo can contain a stale data/meta.json (smoke-test counts). If it is left in
# place, the symlink below is skipped and training would read the wrong token counts.
for f in glob.glob("data/*.bin") + ["data/meta.json"]:
    if os.path.lexists(f):
        os.remove(f)
for f in glob.glob(os.path.join(shard_dir, "*.bin")) + [os.path.join(shard_dir, "meta.json")]:
    os.symlink(f, os.path.join("data", os.path.basename(f)))
print("linked:", sorted(os.listdir("data")))
meta = json.load(open("data/meta.json"))
assert "shards" in meta and "vocab_size" in meta, meta.keys()
for s in meta["shards"]:
    assert {"split", "index", "file", "tokens"} <= set(s), s
print("meta keys OK:", sorted(meta.keys()))
tr = sum(s["tokens"] for s in meta["shards"] if s["split"] == "train")
va = sum(s["tokens"] for s in meta["shards"] if s["split"] == "validation")
print(f"EXACT counts from data/meta.json: train={tr} val={va} vocab={meta['vocab_size']}")
for s in meta["shards"]:
    nbytes = os.path.getsize(os.path.join("data", s["file"]))
    print(f"  {s['file']} {s['tokens']} toks {nbytes/1024/1024:.1f}MB")
    assert nbytes == 2 * s["tokens"], f"{s['file']}: file size does not match meta (stale meta.json?)"
assert tr > 400_000_000, f"train tokens {tr} look wrong (expected ~467M)"
TRAIN_TOKS, VAL_TOKS = tr, va

## Fast tests - fatal on failure (the long Phase 4 tests are skipped on purpose)

`test_inference` also runs its real-SFT-checkpoint gates here (static-cache logits, engine == sampler,
int8 file size), on the kernel's CPU.

In [ ]:
import subprocess, sys

proc = capped([sys.executable, "-m", "pytest", "tests/test_model.py", "tests/test_inference.py",
               "tests/test_speed.py", "-q"])
if proc.returncode != 0:
    raise RuntimeError(f"[bench] fast tests FAILED (rc={proc.returncode}); stopping")
print("[bench] fast tests: PASS", flush=True)


## Benchmark (correctness gate first, then warm-up, interleaved repeats)

`bench.speed` runs the gate before timing: tuple tokens == static tokens, no-cache teacher-forced logits
within 1e-4, engine == bench loop; fp16 reports token agreement (informational). A failed gate raises here.

In [ ]:
import json

budget = int(min(BENCH_BUDGET_S, max(120, time_left() - 8 * 60)))
argv = ["python", "-m", "bench.speed", "--device", "cuda", "--config", "configs/base_30m.yaml",
        "--weights", W, "--prompts", str(PROMPTS), "--gen", str(GEN), "--repeats", str(REPEATS),
        "--warmup", str(WARMUP), "--ns", NS, "--grid", GRID, "--models", MODELS,
        "--budget_s", str(budget), "--out_prefix", "/kaggle/working/benchmark_t4"]
print("[bench] budget_s", budget, "| time left", int(time_left()), flush=True)
proc = capped(argv)
if proc.returncode != 0:
    raise RuntimeError(f"[bench] bench.speed exited {proc.returncode}")
r = json.load(open("/kaggle/working/benchmark_t4_raw.json"))
bad = {k: v for k, v in r["gate"].items() if not v["pass"]}
if bad or r["dropped_by_gate"]:
    raise RuntimeError(f"[bench] correctness gate FAILED: {bad} dropped={r['dropped_by_gate']}")
print("[bench] gate: PASS for every config; repeats completed", r["params"]["repeats"],
      "of", r["params"]["repeats_requested"])
print(open("/kaggle/working/benchmark_t4.md", encoding="utf-8").read())


## Quality numbers - gold-story perplexity, fp32 vs fp16 (same 100 validation prompts as eval_quality)

In [ ]:
import json, math
import numpy as np
import torch
from inference.quantize import to_half
from tokenizer.bpe import BPETokenizer
from train.eval_prompts import val_prompts
from train.eval_quality import mean_nll

tok = BPETokenizer.load(cfg["data"]["tokenizer_path"])
ctx = cfg["train"]["context_len"]
vp = val_prompts(tok, cfg["data"]["out_dir"], 100)
fp32 = GPT(cfg["model"]).cuda().eval()
fp32.load_state_dict(torch.load(W, map_location="cuda", weights_only=True))
fp16 = to_half(fp32)


def gold_ppl(m):
    nll = ntok = 0.0
    with torch.no_grad():
        for _, prompt, _, gold in vp:
            a, b = mean_nll(m, tok, prompt, gold, ctx, "cuda")
            nll, ntok = nll + a * b, ntok + b
    return float(np.exp(nll / ntok))


p32, p16 = gold_ppl(fp32), gold_ppl(fp16)
rel = p16 / p32 - 1
quality = {"gold_ppl_fp32": p32, "gold_ppl_fp16": p16, "fp16_rel": rel, "n_prompts": len(vp),
           "kaggle_sft_reference": 2.636}
json.dump(quality, open("/kaggle/working/quality_t4.json", "w"), indent=1)
print(f"[bench] gold ppl fp32 {p32:.4f} (Kaggle SFT reference 2.636) | fp16 {p16:.4f} ({rel:+.3%})")
COMPLETED = True
print(f"[bench] kernel elapsed {time.time() - T0:.0f}s of the {HARD_CAP_S}s hard cap")


## COMPLETE guard - never claims success after a crash

In [ ]:
if COMPLETED:
    print("=" * 78)
    print("[bench] COMPLETE")
    print("=" * 78, flush=True)
else:
    raise RuntimeError("[bench] COMPLETED sentinel is False; NOT reporting COMPLETE.")
